In [23]:
import torch
import torchvision
from pathlib import Path
import pandas as pd

In [24]:
device = torch.device("cpu")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

In [25]:
BASE_DIR = Path.cwd().parent.parent
TRAIN_DIR = Path(BASE_DIR/'data', 'images', 'train')
VAL_DIR = Path(BASE_DIR/'data', 'images', 'val')
TEST_DIR = Path(BASE_DIR/'data', 'images', 'test')

SPLIT_CSV_PATH = Path(BASE_DIR/'data', 'split.csv')

df = pd.read_csv(SPLIT_CSV_PATH)
df



,filepath,category,defect_type,split
0,test/good/000.png,OK,good,train
1,test/good/001.png,OK,good,train
2,test/good/002.png,OK,good,train
3,test/good/003.png,OK,good,train
4,test/good/004.png,OK,good,test
...,...,...,...,...
475,train/good/315.png,OK,good,train
476,train/good/316.png,OK,good,train
477,train/good/317.png,OK,good,train
478,train/good/318.png,OK,good,test


In [26]:
import sys
sys.path.append(str(BASE_DIR / "src"))
import utils
from utils import ScrewDataset, build_cnn_block, build_model_custom_cnn

In [27]:
df['new_filepath'] = df['filepath'].str.replace(
    r'^([^/]+)/([^/]+)/', r'\1_\2__', regex=True)

In [28]:
cols = ['new_filepath', 'category', 'defect_type']

train_manifest = df.loc[df['split']=='train'][cols]
val_manifest = df.loc[df['split']=='val'][cols]
test_manifest = df.loc[df['split']=='test'][cols]

In [29]:
train_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.RandomHorizontalFlip(p=0.5),
    torchvision.transforms.ColorJitter(brightness=0.1, contrast=0.2),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


eval_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [30]:
train_dataset = ScrewDataset(train_manifest, TRAIN_DIR, train_transform)
val_dataset = ScrewDataset(val_manifest, VAL_DIR, eval_transform)
test_dataset = ScrewDataset(test_manifest, TEST_DIR, eval_transform)

In [31]:
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

In [32]:
model = utils.build_model_custom_cnn(2)
model = model.to(device)

In [33]:
nok_weight = len(train_manifest)/(len(train_manifest[train_manifest['category']=='NOK'])*len(train_manifest['category'].value_counts()))
ok_weight = len(train_manifest)/(len(train_manifest[train_manifest['category']=='OK'])*len(train_manifest['category'].value_counts()))
weight_tensor = torch.tensor([ok_weight, nok_weight], dtype=torch.float32)
criterion = torch.nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = torch.optim.Adam(params=model.parameters(), lr=1e-4)

In [34]:
history, model = utils.train_model(NUM_EPOCHS=250, PATIENCE=250, model=model, train_loader=train_loader, val_loader=val_loader, criterion=criterion, optimizer=optimizer, device=device)

In [35]:
data = pd.DataFrame(history)
data

,epoch,train_loss,val_loss,val_accuracy
0,1,0.705140,0.683762,0.680556
1,2,0.704333,0.685346,0.513889
2,3,0.670220,0.653967,0.736111
3,4,0.673245,0.653522,0.777778
4,5,0.698132,0.684269,0.430556
...,...,...,...,...
245,246,0.183701,2.663813,0.263889
246,247,0.166293,0.825351,0.791667
247,248,0.157511,0.506747,0.763889
248,249,0.144843,0.401589,0.847222


In [36]:
torch.save(model.state_dict(), Path(BASE_DIR, 'models', 'custom_cnn', 'best_lr_4.pth'))

In [37]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["OK", "NOK"]))

[[48  6]
 [ 6 12]]
              precision    recall  f1-score   support

          OK       0.89      0.89      0.89        54
         NOK       0.67      0.67      0.67        18

    accuracy                           0.83        72
   macro avg       0.78      0.78      0.78        72
weighted avg       0.83      0.83      0.83        72



In [1]:
import matplotlib.pyplot as plt

df = pd.DataFrame(history)
best_epoch = df.loc[df['val_loss'].idxmin(), 'epoch']

plt.figure(figsize=(8, 5))
plt.plot(df['epoch'], df['train_loss'], label='train_loss')
plt.plot(df['epoch'], df['val_loss'], label='val_loss')
plt.axvline(best_epoch, color='gray', linestyle='--', label=f'best epoch ({best_epoch})')
plt.xlabel('Epoka')
plt.ylabel('Strata')
plt.title('Krzywe uczenia — custom CNN(MVTec)')
plt.legend()
plt.show()

NameError: name 'pd' is not defined

In [39]:
sum(p.numel() for p in model.parameters())

97938